# 🧮 Qwen2.5-1.5B Math SFT Training Pipeline
### DeepSeek-R1 Reproduction at Small Scale

**Pipeline:**
1. Install dependencies
2. Load Qwen2.5-1.5B base model
3. Evaluate baseline accuracy on GSM8K (pre-SFT)
4. Load NuminaMath-CoT dataset & fine-tune with SFT
5. Evaluate accuracy post-SFT
6. Compare model outputs: before vs after

> **GPU:** Designed for Google Colab T4 (15GB VRAM) using 4-bit quantization + LoRA (QLoRA)

## 📦 Step 1: Install Dependencies

In [ ]:
%%capture
!pip install -q torch==2.3.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.46.3
!pip install -q trl==0.12.1
!pip install -q peft==0.13.2
!pip install -q bitsandbytes==0.43.3   # ← downgrade from 0.44.1
!pip install -q accelerate==1.1.1
!pip install -q datasets==3.1.0
!pip install -q triton==2.3.0          # ← pin triton explicitly
!pip install -q sentencepiece matplotlib seaborn
print('✅ All packages installed!')

In [ ]:
import torch, gc, re, json, warnings
warnings.filterwarnings('ignore')

print(f'🖥️  Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'🎮 GPU: {gpu.name}')
    print(f'💾 VRAM: {gpu.total_memory / 1e9:.1f} GB')
    print(f'🔧 CUDA: {torch.version.cuda}')

🖥️  Device: CUDA
🎮 GPU: Tesla T4
💾 VRAM: 15.6 GB
🔧 CUDA: 12.1


## 🔧 Step 2: Configuration

In [ ]:
# ═══════════════════════════════════════════════
#  CONFIG — tweak these to your needs
# ═══════════════════════════════════════════════
MODEL_ID       = "Qwen/Qwen2.5-1.5B"
DATASET_ID     = "AI-MO/NuminaMath-CoT"
OUTPUT_DIR     = "./qwen-math-sft"

# Eval config
EVAL_SAMPLES   = 50     # GSM8K samples to evaluate on
MAX_NEW_TOKENS = 256    # Tokens to generate during eval

# SFT training config (T4-safe)
SFT_SAMPLES    = 2000   # NuminaMath samples to train on
EPOCHS         = 1
BATCH_SIZE     = 2
GRAD_ACCUM     = 8      # Effective batch = 16
LR             = 2e-4
MAX_SEQ_LEN    = 512
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05

print('⚙️  Config loaded!')
print(f'   Model:        {MODEL_ID}')
print(f'   Dataset:      {DATASET_ID}')
print(f'   Eval samples: {EVAL_SAMPLES}')
print(f'   SFT samples:  {SFT_SAMPLES}')

⚙️  Config loaded!
   Model:        Qwen/Qwen2.5-1.5B
   Dataset:      AI-MO/NuminaMath-CoT
   Eval samples: 50
   SFT samples:  2000


## 🤖 Step 3: Load Base Model (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

from transformers import AutoTokenizer, AutoModelForCausalLM

print('📥 Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('📥 Loading model (no quantization)...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    # No quantization_config here
)
base_model.config.use_cache = False

params = sum(p.numel() for p in base_model.parameters()) / 1e9
print(f'✅ Model loaded ({params:.2f}B params)')

📥 Loading tokenizer...
📥 Loading model (no quantization)...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ Model loaded (1.54B params)


## 📊 Step 4: Baseline Evaluation (Pre-SFT)

Evaluate zero-shot performance on GSM8K using exact answer match.

In [ ]:
from datasets import load_dataset

print('📥 Loading GSM8K test set...')
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
eval_data = gsm8k.select(range(EVAL_SAMPLES))
print(f'✅ {len(eval_data)} evaluation samples loaded')
print(f'\nSample — Q: {eval_data[0]["question"][:80]}...')
print(f'          A: {eval_data[0]["answer"][-40:]}')

📥 Loading GSM8K test set...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

✅ 50 evaluation samples loaded

Sample — Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an...
          A: very day at the farmer’s market.
#### 18


In [ ]:
def extract_answer(text):
    """Pull a numeric answer out of model output or GSM8K label."""
    # GSM8K ground-truth format:  #### 42
    m = re.search(r'####\s*([\-\d,\.]+)', text)
    if m:
        return m.group(1).replace(',', '').strip()
    # Model output patterns
    for pat in [
        r'[Tt]he answer is[:\s]+([\-\d,\.]+)',
        r'[Aa]nswer[:\s=]+([\-\d,\.]+)',
        r'=\s*([\-\d,\.]+)\s*$',
        r'([\-\d,\.]+)\s*$',
    ]:
        m = re.search(pat, text.strip())
        if m:
            return m.group(1).replace(',', '').strip()
    return None


def build_prompt(question):
    return (
        "Solve the following math problem step by step. "
        "At the end, state the final answer clearly.\n\n"
        f"Problem: {question}\n\nSolution:"
    )


@torch.inference_mode()
def evaluate_model(model, tokenizer, eval_data, desc="Evaluating", store_outputs=False):
    model.eval()
    correct, results = 0, []
    for i, sample in enumerate(eval_data):
        gt_answer = extract_answer(sample['answer'])
        inputs = tokenizer(
            build_prompt(sample['question']),
            return_tensors='pt', truncation=True, max_length=512
        ).to(model.device)
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()
        pred = extract_answer(generated)
        ok = (pred == gt_answer) if (pred and gt_answer) else False
        if ok: correct += 1
        if store_outputs:
            results.append({'question': sample['question'], 'gt_answer': gt_answer,
                            'pred_answer': pred, 'generated_text': generated, 'correct': ok})
        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{len(eval_data)}] running accuracy: {correct/(i+1)*100:.1f}%')
    acc = correct / len(eval_data) * 100
    print(f'\n✅ {desc} — Accuracy: {acc:.1f}% ({correct}/{len(eval_data)})')
    return acc, results


print('🔍 Evaluating BASE model (before SFT)...')
print('='*55)
baseline_acc, baseline_outputs = evaluate_model(
    base_model, tokenizer, eval_data, desc="Baseline (pre-SFT)", store_outputs=True
)

🔍 Evaluating BASE model (before SFT)...
  [10/50] running accuracy: 0.0%
  [20/50] running accuracy: 10.0%
  [30/50] running accuracy: 6.7%
  [40/50] running accuracy: 7.5%
  [50/50] running accuracy: 8.0%

✅ Baseline (pre-SFT) — Accuracy: 8.0% (4/50)


## 👀 Step 5: Inspect Baseline Outputs

In [ ]:
print('='*65)
print('BASELINE MODEL — Sample Outputs (Pre-SFT)')
print('='*65)
for i, r in enumerate(baseline_outputs[:3]):
    status = '✅ CORRECT' if r['correct'] else '❌ WRONG'
    print(f'\n── Sample {i+1} {status} ──')
    q = r['question']
    print(f'📝 Question: {q[:120]}...' if len(q)>120 else f'📝 Question: {q}')
    print(f'🎯 Ground truth: {r["gt_answer"]}')
    print(f'🤖 Predicted:    {r["pred_answer"]}')
    g = r['generated_text']
    print(f'📄 Raw output:\n{g[:400]}...' if len(g)>400 else f'📄 Raw output:\n{g}')
    print()

BASELINE MODEL — Sample Outputs (Pre-SFT)

── Sample 1 ❌ WRONG ──
📝 Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every da...
🎯 Ground truth: 18
🤖 Predicted:    .
📄 Raw output:
Step 1: Calculate the total number of eggs laid per day.
Janet's ducks lay 16 eggs per day.

Step 2: Calculate the number of eggs eaten for breakfast.
Janet eats 3 eggs for breakfast every morning.

Step 3: Calculate the number of eggs used for muffins.
Janet uses 4 eggs to bake muffins for her friends.

Step 4: Calculate the number of eggs left for the farmers' market.
Subtract the number of eggs...


── Sample 2 ❌ WRONG ──
📝 Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
🎯 Ground truth: 3
🤖 Predicted:    .
📄 Raw output:
Step 1: Identify the amount of blue fiber needed for the robe.
The problem states that the robe takes 2 bolts of blue fiber.

Step 2: Identify the amo

## 📚 Step 6: Load & Prepare SFT Dataset (NuminaMath-CoT)

In [ ]:
print(f'📥 Loading NuminaMath-CoT ({SFT_SAMPLES} samples)...')
numina = load_dataset(DATASET_ID, split="train")
numina_subset = numina.select(range(SFT_SAMPLES))
print(f'✅ Loaded. Features: {list(numina_subset.features.keys())}')
s = numina_subset[0]
print(f'\nSample problem: {s["problem"][:100]}...')
print(f'Sample solution: {s["solution"][:150]}...')

📥 Loading NuminaMath-CoT (2000 samples)...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

...

data/train-00004-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/166k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/859494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Loaded. Features: ['source', 'problem', 'solution', 'messages']

Sample problem: Consider the terms of an arithmetic sequence: $-\frac{1}{3}, y+2, 4y, \ldots$. Solve for $y$....
Sample solution: For an arithmetic sequence, the difference between consecutive terms must be equal. Therefore, we can set up the following equations based on the sequ...


In [ ]:
def format_sft_example(example):
    """Format NuminaMath into instruction text with <think> tags."""
    boxed = re.search(r'\\boxed\{([^}]+)\}', example['solution'])
    final_ans = boxed.group(1) if boxed else 'See solution'
    prompt = (
        "Solve the following math problem step by step. "
        "Show your reasoning, then state the final answer.\n\n"
        f"Problem: {example['problem']}\n\nSolution:"
    )
    response = f" <think>\n{example['solution']}\n</think>\nFinal Answer: {final_ans}"
    full_text = prompt + response
    tok = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LEN, padding='max_length')
    tok['labels'] = tok['input_ids'].copy()
    return tok


print('🔄 Tokenizing dataset...')
tokenized_dataset = numina_subset.map(
    format_sft_example,
    remove_columns=numina_subset.column_names,
    desc='Tokenizing', num_proc=1,
)
tokenized_dataset.set_format('torch')
print(f'✅ Tokenized {len(tokenized_dataset)} samples')

🔄 Tokenizing dataset...


Tokenizing:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Tokenized 2000 samples


## 🏋️ Step 7: LoRA Setup + SFT Training

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# No prepare_model_for_kbit_training since we're not quantizing
base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.CAUSAL_LM,
)
peft_model = get_peft_model(base_model, lora_config)

for name, param in peft_model.named_parameters():
    if "lora" in name:
        param.requires_grad_(True)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in peft_model.parameters())
print(f'✅ LoRA applied: {trainable/1e6:.1f}M / {total/1e6:.0f}M params trainable ({trainable/total*100:.2f}%)')

✅ LoRA applied: 18.5M / 1562M params trainable (1.18%)


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, TrainerCallback
import os

class DriveCheckpointCallback(TrainerCallback):
    def __init__(self, drive_dir, save_every_n_epochs=5):
        self.drive_dir = drive_dir
        self.save_every = save_every_n_epochs

    def on_epoch_end(self, args, state, control, model=None, tokenizer=None, **kwargs):
        epoch = int(state.epoch)
        if epoch % self.save_every == 0:
            ckpt_path = os.path.join(self.drive_dir, f"checkpoint-epoch-{epoch}")
            os.makedirs(ckpt_path, exist_ok=True)
            print(f"\n💾 Saving checkpoint at epoch {epoch} → {ckpt_path}")
            model.save_pretrained(ckpt_path)
            if tokenizer is not None:
                tokenizer.save_pretrained(ckpt_path)
            print(f"✅ Checkpoint saved!")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # ← critical fix
    optim="adamw_torch",          # ← changed from paged_adamw_8bit (needs bitsandbytes)
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=torch.cuda.is_available(),   # fp16 only if GPU available
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

drive_callback = DriveCheckpointCallback(
    drive_dir="/content/drive/MyDrive/RL Assignment",
    save_every_n_epochs=5,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[drive_callback],
)

print('🚀 Starting SFT training...')
print(f'   Samples:  {len(tokenized_dataset)}')
print(f'   Epochs:   {EPOCHS}')
print(f'   LR:       {LR}')
print(f'   Drive checkpoints every 5 epochs → /content/drive/MyDrive/RL Assignment')
print('='*55)

train_result = trainer.train()

print(f'\n✅ Training complete!')
print(f'   Train loss:    {train_result.training_loss:.4f}')
print(f'   Train runtime: {train_result.metrics["train_runtime"]/60:.1f} min')

final_path = os.path.join("/content/drive/MyDrive/RL Assignment", "final")
os.makedirs(final_path, exist_ok=True)
peft_model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'💾 Final model saved locally  → {OUTPUT_DIR}')
print(f'💾 Final model saved to Drive → {final_path}')

🚀 Starting SFT training...
   Samples:  2000
   Epochs:   1
   LR:       0.0002
   Drive checkpoints every 5 epochs → /content/drive/MyDrive/RL Assignment


Step,Training Loss
50,0.578700
100,0.525800



✅ Training complete!
   Train loss:    0.5458
   Train runtime: 16.7 min
💾 Final model saved locally  → ./qwen-math-sft
💾 Final model saved to Drive → /content/drive/MyDrive/RL Assignment/final


Save required Stuff on drive

In [ ]:
import os
import json
import torch

DRIVE_DIR = "/content/drive/MyDrive/RL Assignment"

# ── 1. LoRA adapters (most important) ────────────────────────
adapter_path = os.path.join(DRIVE_DIR, "sft_lora_adapters")
peft_model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"✅ LoRA adapters saved → {adapter_path}")

# ── 2. Training config (so you remember what you used) ────────
config = {
    "model_id":       MODEL_ID,
    "dataset_id":     DATASET_ID,
    "sft_samples":    SFT_SAMPLES,
    "epochs":         EPOCHS,
    "batch_size":     BATCH_SIZE,
    "grad_accum":     GRAD_ACCUM,
    "lr":             LR,
    "max_seq_len":    MAX_SEQ_LEN,
    "lora_r":         LORA_R,
    "lora_alpha":     LORA_ALPHA,
    "lora_dropout":   LORA_DROPOUT,
    "final_loss":     train_result.training_loss,
    "runtime_min":    train_result.metrics["train_runtime"] / 60,
}
with open(os.path.join(DRIVE_DIR, "sft_config.json"), "w") as f:
    json.dump(config, f, indent=2)
print(f"✅ Config saved → sft_config.json")

# ── 3. Training loss history (for plotting later) ─────────────
loss_history = [
    {"step": log["step"], "loss": log["loss"]}
    for log in trainer.state.log_history
    if "loss" in log
]
with open(os.path.join(DRIVE_DIR, "sft_loss_history.json"), "w") as f:
    json.dump(loss_history, f, indent=2)
print(f"✅ Loss history saved → sft_loss_history.json")

print("\n🎉 All artifacts saved! Safe to disconnect.")